# 📘 Tutorial 10: Mini-project — Raw CSV to Plot-ready Table

> This notebook is provided in a clean, non-executed state for readability and reproducibility.

This capstone pulls together the Part 1 workflow. We start with a small raw CSV, inspect it, clean numeric columns, add derived columns, join metadata, aggregate replicates, and export a final plot-ready table.

---

## 🎯 Learning objectives

By the end of this notebook, you should be able to:

- load a small raw dataset,
- inspect structure and dtypes,
- clean numeric columns,
- add derived response columns,
- join metadata,
- aggregate replicates,
- create a final plot-ready table,
- export processed data for a later Matplotlib workflow.


## 🔧 Setup


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


## 1. Locate the project data


In [ ]:
project_root = Path.cwd()
if not (project_root / "data" / "part1").exists():
    project_root = project_root.parent

data_dir = project_root / "data" / "part1"
processed_dir = data_dir / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

raw_path = data_dir / "capstone_raw_measurements.csv"
metadata_path = data_dir / "capstone_sample_metadata.csv"


## 2. Load and inspect the raw data


In [ ]:
raw = pd.read_csv(raw_path)
raw.head()


In [ ]:
print("Shape:", raw.shape)
print("Columns:", raw.columns.to_list())
raw.dtypes


In [ ]:
raw.tail()


The `blank` and `signal` columns look numeric, but they were read as object columns because the raw file contains invalid or missing entries.


## 3. Clean numeric columns


In [ ]:
clean = raw.copy()

numeric_columns = ["replicate", "time_min", "wavelength_nm", "blank", "signal"]
for column in numeric_columns:
    clean[column] = pd.to_numeric(clean[column], errors="coerce")

clean.dtypes


In [ ]:
clean.isna().sum()


We now have true numeric columns and visible missing values.


## 4. Apply simple validity rules


In [ ]:
invalid_signal = (clean["signal"] < 0) | (clean["signal"] > 2)
invalid_blank = (clean["blank"] < 0) | (clean["blank"] > 0.2)

clean["review_note"] = ""
clean.loc[invalid_signal, "review_note"] = "signal outside expected range"
clean.loc[invalid_blank, "review_note"] = "blank outside expected range"

clean.loc[invalid_signal, "signal"] = np.nan
clean.loc[invalid_blank, "blank"] = np.nan

clean.loc[clean["review_note"] != ""]


In a real project, validity ranges should be justified by the instrument, assay, or domain context. Here they simply make the cleaning decision explicit.


## 5. Add derived columns


In [ ]:
clean["net_response"] = clean["signal"] - clean["blank"]
clean[["sample_id", "replicate", "time_min", "blank", "signal", "net_response"]].head()


In [ ]:
clean_rows = clean.dropna(subset=["time_min", "signal", "blank", "net_response"]).copy()
clean_rows.shape


Rows with missing values in columns needed for plotting are removed from the cleaned plotting workflow. The raw table remains unchanged.


## 6. Join metadata


In [ ]:
metadata = pd.read_csv(metadata_path)
metadata


In [ ]:
joined = clean_rows.merge(
    metadata,
    on="sample_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

joined["_merge"].value_counts()


In [ ]:
assert (joined["_merge"] == "both").all()
joined = joined.drop(columns="_merge")
joined.head()


## 7. Aggregate replicates


In [ ]:
replicate_summary = (
    joined.groupby(
        ["condition", "concentration_mM", "time_min"],
        as_index=False,
    )
    .agg(
        mean_net_response=("net_response", "mean"),
        sd_net_response=("net_response", "std"),
        n=("net_response", "count"),
    )
    .sort_values(["concentration_mM", "time_min"])
)

replicate_summary["sem_net_response"] = (
    replicate_summary["sd_net_response"] / (replicate_summary["n"] ** 0.5)
)

replicate_summary


This table is now plot-ready for a time-course comparison by condition.


## 8. Create x/y arrays for one condition


In [ ]:
high_condition = replicate_summary.loc[
    replicate_summary["condition"] == "high"
].sort_values("time_min")

x = high_condition["time_min"].to_numpy()
y = high_condition["mean_net_response"].to_numpy()

print("x:", x)
print("y:", y)


This is where a later Matplotlib notebook would take over: choose figure style, axes, labels, markers, error bars, and export settings.


## 9. Export the final plot-ready table


In [ ]:
final_plot_ready = replicate_summary[
    [
        "condition",
        "concentration_mM",
        "time_min",
        "mean_net_response",
        "sem_net_response",
        "n",
    ]
].copy()

output_path = processed_dir / "capstone_plot_ready_summary.csv"
final_plot_ready.to_csv(output_path, index=False)

print(output_path)
final_plot_ready


In [ ]:
reloaded = pd.read_csv(output_path)
assert list(reloaded.columns) == list(final_plot_ready.columns)
assert len(reloaded) == len(final_plot_ready)


## 10. What comes next in Matplotlib

A plotting tutorial could now focus on:

- drawing one line per condition,
- adding error bars from `sem_net_response`,
- controlling colours and markers,
- labelling axes with units,
- exporting publication-ready figures.

The data work is already explicit and reproducible.


## 🧭 Key takeaways

- A raw CSV to plot-ready workflow has several deliberate stages.
- Inspection comes before cleaning.
- Numeric conversion should expose invalid values rather than hide them.
- Metadata joins should be validated.
- Replicate summaries are useful outputs but should not erase raw data.
- Exported plot-ready tables make later plotting notebooks cleaner.
